In [ ]:
import pandas as pd

# Read the Bronze Parquet file
bronze_path = "/lakehouse/default/Files/bronze_cms_payments.parquet"
df = pd.read_parquet(bronze_path)

print(f"Bronze records loaded: {len(df)}")
print(f"Columns: {df.shape[1]}")
display(df.head(3))

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
# SILVER TRANSFORMATIONS
# 1. Select the most useful columns (drop redundant/empty ones)
# 2. Rename to clean names
# 3. Fix data types
# 4. Flag missing critical fields

silver_cols = [
    'covered_recipient_type',
    'covered_recipient_first_name',
    'covered_recipient_last_name',
    'covered_recipient_npi',
    'covered_recipient_specialty_1',
    'recipient_city',
    'recipient_state',
    'submitting_applicable_manufacturer_or_applicable_gpo_name',
    'total_amount_of_payment_usdollars',
    'date_of_payment',
    'form_of_payment_or_transfer_of_value',
    'nature_of_payment_or_transfer_of_value',
    'program_year'
]

df_silver = df[silver_cols].copy()

# Rename for readability
df_silver.columns = [
    'recipient_type',
    'first_name',
    'last_name',
    'npi',
    'specialty',
    'city',
    'state',
    'manufacturer_name',
    'payment_amount',
    'payment_date',
    'payment_form',
    'payment_nature',
    'program_year'
]

# Fix data types
df_silver['payment_amount'] = pd.to_numeric(df_silver['payment_amount'], errors='coerce')
df_silver['payment_date'] = pd.to_datetime(df_silver['payment_date'], errors='coerce')
df_silver['program_year'] = df_silver['program_year'].astype(str)

# Flag missing critical fields
df_silver['data_quality_flag'] = (
    df_silver['npi'].isna() | 
    df_silver['payment_amount'].isna() |
    df_silver['manufacturer_name'].isna()
)

print(f"Silver records: {len(df_silver)}")
print(f"Columns: {df_silver.shape[1]}")
print(f"Data quality flags: {df_silver['data_quality_flag'].sum()}")
print(f"Payment amount range: ${df_silver['payment_amount'].min():,.2f} - ${df_silver['payment_amount'].max():,.2f}")
display(df_silver.head(3))

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# Write Silver data to Lakehouse
silver_path = "/lakehouse/default/Files/silver_cms_payments.parquet"
df_silver.to_parquet(silver_path, index=False)

print(f"Successfully written {len(df_silver)} records to Silver Lakehouse")
print(f"Location: Files/silver_cms_payments.parquet")
print(f"Columns: {list(df_silver.columns)}")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# Write all analytics tables directly to Lakehouse Tables (not Files)
# This makes them available to Power BI automatically

df_silver.to_parquet("/lakehouse/default/Files/gold_by_manufacturer.parquet", index=False)

# Aggregations
gold_manufacturer = df_silver.groupby('manufacturer_name').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

gold_state = df_silver.groupby('state').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

gold_specialty = df_silver.groupby('specialty').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

gold_manufacturer.to_parquet("/lakehouse/default/Files/gold_by_manufacturer.parquet", index=False)
gold_state.to_parquet("/lakehouse/default/Files/gold_by_state.parquet", index=False)
gold_specialty.to_parquet("/lakehouse/default/Files/gold_by_specialty.parquet", index=False)

print("Gold files written to Bronze Lakehouse:")
print(f"  Manufacturers: {len(gold_manufacturer)} rows")
print(f"  States: {len(gold_state)} rows")
print(f"  Specialties: {len(gold_specialty)} rows")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [1]:
import pandas as pd
import os

# Read parquet files
gold_manufacturer = pd.read_parquet("/lakehouse/default/Files/gold_by_manufacturer.parquet")
gold_state = pd.read_parquet("/lakehouse/default/Files/gold_by_state.parquet")
gold_specialty = pd.read_parquet("/lakehouse/default/Files/gold_by_specialty.parquet")
silver = pd.read_parquet("/lakehouse/default/Files/silver_cms_payments.parquet")

# Write as CSV files - Power BI can read these directly
gold_manufacturer.to_csv("/lakehouse/default/Files/gold_by_manufacturer.csv", index=False)
gold_state.to_csv("/lakehouse/default/Files/gold_by_state.csv", index=False)
gold_specialty.to_csv("/lakehouse/default/Files/gold_by_specialty.csv", index=False)
silver.to_csv("/lakehouse/default/Files/silver_cms_payments.csv", index=False)

print("CSV files written:")
print(os.listdir('/lakehouse/default/Files'))

CSV files written:
['gold_by_manufacturer.csv', 'gold_by_specialty.parquet', 'gold_by_specialty.csv', 'gold_by_manufacturer.parquet', 'gold_by_state.parquet', 'silver_cms_payments.parquet', 'silver_cms_payments.csv', 'gold_by_state.csv', 'bronze_cms_payments.parquet']
